# A.3 文件多了怎麼找相關段落？


十份公告可以逐份讀，一萬份公告就需要先縮小範圍。問「書店地址」時，希望先取出有地址的段落，而不是把所有文件塞進模型。這一步叫retrieval（檢索）：依問題排序文件，挑出若幹候選。前置是[RAG的找資料與回答兩階段](https://birdhackor.github.io/tiny-perceptron-vlm/A.2.html)。本節先用能逐項核對的字詞匹配，理解檢索器到底憑什麼選文件。

查詢「書店地址」有書、店、地、址四個字。文件d1「貓喜歡曬太陽」一個也沒有，分數0；d2「書店地址是青街8號」四個都有，分數4。分數不是回答正確的機率，只是這套規則算出的重合數。它不理解青街8號是地址，也不知道文件是否最新，但能把d2排在d1前面。

工具的`lexical_terms`先把英文轉小寫，中文按單字拆開，英文與數字按連續片段拆開。因此「Book shop」成為book、shop；中文的書店成為書、店，並沒有使用語言模型理解完整詞義。`Counter`記錄每個片段出現次數；每項貢獻取「查詢次數與文件次數中較小者」，再加總。查詢只出現一次「書」，文件重複十次「書」仍只貢獻1，不會變成10分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.retrieval import lexical_terms, retrieve

query = "書店地址"
docs = [
    {"id": "d1", "text": "貓喜歡曬太陽"},
    {"id": "d2", "text": "書店地址是青街8號"},
]
print("查詢片段", lexical_terms(query))
hits = retrieve(query, docs, k=1)
print("取回", hits)
print("改問法", retrieve("營業處在哪", docs, k=1))

輸入是查詢字串與兩份含id、text的文件，`k=1`表示最多取回一份。第一行應印四個中文字片段，第二行只含d2及其原文。最後一行是空清單：營、業、處、在、哪與兩份原文沒有重合，雖然人可能猜到在問營業地點，這個檢索器沒有同義詞知識。零分文件不會回傳；同分時按id排序，結果也不一定剛好是最有用的文件。

更實用的檢索可以考慮不同詞的重要性，或把問題與段落轉成數字特徵，比較語意接近程度。但換方法前，仍應用一組有標準來源的問題量「是否找到所需資料」。取回段落只完成RAG的第一步，生成模型還得讀對，來源還得可信。像「書店地址尚未公布」也會拿到4分，卻沒有可回答的地址，這是匹配分數的界限。

練習只把d2的原文改成「書店地址尚未公布」，保持查詢與k不變。先預測仍會取回d2，再執行核對；然後用一句話說明為何這份hit不能支持「青街8號」。你要分開判斷「被檢索器選中」與「足夠回答問題」，不能從非空清單直接跳到答案。
